# 02 — Feature Leakage Audit

This notebook defines **prediction time** and audits every raw feature in the Bank Marketing dataset against it, to decide which features are legitimately available for a realistic predictive model and which would leak information from the future. It does not clean data, engineer features, or train any model — it only produces an audit table and a recommended primary feature set for later notebooks to use.

## What is prediction-time leakage?

We define **prediction time** as the moment **immediately before a scheduled customer contact begins** — i.e. the bank has already decided to call this client, on a known date and through a known channel, but has not yet placed that call. A feature is only legitimate if its value would actually be known and fixed at that moment.

Under this definition, the *scheduling* of a contact (when it will happen, and through which channel) is legitimate planning information set in advance — it is not a byproduct of the call's outcome. What remains illegitimate is anything determined *during or after* the call itself (e.g. how long it lasted) or that depends on the cumulative outcome of contact attempts within the current campaign.

Prediction-time leakage happens when a feature's value is only determined *during* or *after* the event being predicted (the client's response to this call). Such a feature can look highly predictive in historical data purely because it is a byproduct of the outcome itself — not because it causally precedes it. A model trained on it would be unusable in production, since those values don't exist yet when a real prediction is needed.

In [ ]:
import os
import pandas as pd

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

pd.set_option("display.max_colwidth", 100)

## Load the dataset (read-only)

We only read the raw columns to confirm the audit table below matches the actual schema — the dataset itself is not modified, filtered, or saved anywhere in this notebook.

In [ ]:
CSV_NAME = "bank-additional-full.csv"

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

_candidates = [
    f"../data/{CSV_NAME}",
    f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore

        print(f"'{CSV_NAME}' not found in: {_candidates}")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df = pd.read_csv(DATA_PATH, sep=";")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
list(df.columns)

## Feature availability audit

Each raw feature is classified into one of four groups:

- **Legitimate — customer profile**: static or slow-changing attributes of the client, known independently of any campaign activity.
- **Legitimate — previous campaign history**: outcomes/timing from *prior* campaigns, which are historical record by the time the current campaign starts.
- **Legitimate — scheduled contact plan**: `contact`, `month`, `day_of_week` describe *when* and *how* the call is planned to happen — this is decided by the bank in advance and is known immediately before the call is placed, so it is included under the revised prediction-time definition.
- **Excluded — in-call / cumulative campaign information**: `duration` (only known once the call ends) and `campaign` (a running count of contacts for the current campaign, not fixed until the campaign concludes for that client).
- **Excluded — economic indicators**: `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`. These are macroeconomic, monthly-level indicators. While conceptually they *could* be known ahead of time, the dataset does not make clear whether each record's value was the one truly available at that client's individual prediction time (vs. snapshotted alongside the campaign/call record) — so we exclude them from the primary feature set until that record-level availability is verified. The target `y` is excluded as it is the outcome being predicted, not a feature.

The table below lists every raw column with a description, its availability relative to prediction time, the audit decision, and the justification.

In [ ]:
audit_rows = [
    # --- Customer profile: legitimate ---
    dict(feature="age", description="Client's age in years.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Static client attribute, known independently of any campaign."),
    dict(feature="job", description="Type of job.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Static client attribute, known independently of any campaign."),
    dict(feature="marital", description="Marital status.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Static client attribute, known independently of any campaign."),
    dict(feature="education", description="Education level.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Static client attribute, known independently of any campaign."),
    dict(feature="default", description="Has credit in default?",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Client credit-record attribute, known from existing bank records before the campaign."),
    dict(feature="housing", description="Has a housing loan?",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Client credit-record attribute, known from existing bank records before the campaign."),
    dict(feature="loan", description="Has a personal loan?",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Client credit-record attribute, known from existing bank records before the campaign."),

    # --- Previous campaign history: legitimate ---
    dict(feature="pdays", description="Days since the client was last contacted in a previous campaign (999 = never contacted).",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Describes a previous campaign, which is complete and historical by the time the current campaign starts."),
    dict(feature="previous", description="Number of contacts performed before this campaign, for this client.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Describes a previous campaign, which is complete and historical by the time the current campaign starts."),
    dict(feature="poutcome", description="Outcome of the previous marketing campaign.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="Describes a previous campaign, which is complete and historical by the time the current campaign starts."),

    # --- Scheduled contact plan: legitimate ---
    dict(feature="contact", description="Contact communication type used for the scheduled call.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="The communication channel is decided by the bank when scheduling the contact, so it is known immediately before the call is placed."),
    dict(feature="month", description="Scheduled contact month.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="The contact date is set by the bank's schedule in advance, so it is known immediately before the call is placed."),
    dict(feature="day_of_week", description="Scheduled contact day of week.",
         availability_before_current_campaign="Known",
         decision="Include",
         justification="The contact date is set by the bank's schedule in advance, so it is known immediately before the call is placed."),

    # --- In-call / cumulative campaign information: excluded ---
    dict(feature="duration", description="Last contact duration, in seconds, for the current campaign.",
         availability_before_current_campaign="Not known",
         decision="Exclude",
         justification="Only known after the call ends, and near-perfectly correlated with the outcome itself (e.g. 0 seconds implies y='no') — the canonical leakage feature in this dataset."),
    dict(feature="campaign", description="Number of contacts performed during this campaign, for this client (includes the last contact).",
         availability_before_current_campaign="Not known",
         decision="Exclude",
         justification="Accumulates contact attempts made during the current campaign; its final value is only known once the campaign (for this client) has concluded, not before any single scheduled call."),

    # --- Economic indicators: excluded (uncertain record-level availability) ---
    dict(feature="emp.var.rate", description="Employment variation rate (quarterly macro indicator).",
         availability_before_current_campaign="Uncertain",
         decision="Exclude",
         justification="Macroeconomic indicator; record-level alignment with each client's true prediction time is not verifiable from the dataset, so it is held out of the primary feature set pending further verification."),
    dict(feature="cons.price.idx", description="Consumer price index (monthly macro indicator).",
         availability_before_current_campaign="Uncertain",
         decision="Exclude",
         justification="Macroeconomic indicator; record-level alignment with each client's true prediction time is not verifiable from the dataset, so it is held out of the primary feature set pending further verification."),
    dict(feature="cons.conf.idx", description="Consumer confidence index (monthly macro indicator).",
         availability_before_current_campaign="Uncertain",
         decision="Exclude",
         justification="Macroeconomic indicator; record-level alignment with each client's true prediction time is not verifiable from the dataset, so it is held out of the primary feature set pending further verification."),
    dict(feature="euribor3m", description="Euribor 3-month rate (daily macro indicator).",
         availability_before_current_campaign="Uncertain",
         decision="Exclude",
         justification="Macroeconomic indicator; record-level alignment with each client's true prediction time is not verifiable from the dataset, so it is held out of the primary feature set pending further verification."),
    dict(feature="nr.employed", description="Number of employees (quarterly macro indicator).",
         availability_before_current_campaign="Uncertain",
         decision="Exclude",
         justification="Macroeconomic indicator; record-level alignment with each client's true prediction time is not verifiable from the dataset, so it is held out of the primary feature set pending further verification."),

    # --- Target: not a feature ---
    dict(feature="y", description="Has the client subscribed to a term deposit? (target variable)",
         availability_before_current_campaign="Not applicable",
         decision="Target (not a feature)",
         justification="This is the label being predicted, not an input feature."),
]

audit_df = pd.DataFrame(audit_rows, columns=[
    "feature", "description", "availability_before_current_campaign", "decision", "justification",
])
audit_df

In [ ]:
# Sanity check: audit table must cover every column in the actual dataset, and nothing extra.
assert set(audit_df["feature"]) == set(df.columns), (
    set(df.columns).symmetric_difference(set(audit_df["feature"]))
)
print("Audit table covers all", len(df.columns), "dataset columns.")

## Save the audit table

Saved as a standalone artifact of this audit — not the dataset itself, which remains unmodified.

In [ ]:
audit_output_path = f"{RESULTS_DIR}/feature_availability_audit.csv"
audit_df.to_csv(audit_output_path, index=False)
print("Saved audit table to:", audit_output_path)

## Final primary feature set

Based on the revised audit above, the primary feature set for a model predicting immediately before a scheduled customer contact includes customer-profile, previous-campaign-history, and scheduled-contact-plan features:

- **Customer profile**: `age`, `job`, `marital`, `education`, `default`, `housing`, `loan`
- **Previous campaign history**: `pdays`, `previous`, `poutcome`
- **Scheduled contact plan**: `contact`, `month`, `day_of_week`

`duration` and `campaign` remain excluded — both are only fully determined during or after the current campaign's contact attempts, not before a scheduled call. The economic indicators (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`) also remain excluded from the primary set due to uncertainty about their exact pre-call, record-level availability. No columns are dropped from the dataset itself in this notebook — this list is a specification for feature selection to be applied in a later preprocessing/modeling notebook.

In [ ]:
PRIMARY_FEATURE_SET = audit_df.loc[audit_df["decision"] == "Include", "feature"].tolist()
print(f"Primary feature set ({len(PRIMARY_FEATURE_SET)} features):")
PRIMARY_FEATURE_SET